══════════════════════════════════════════════════════════════
 WEEK 4  |  DAY 3  |  DATA CLEANING

══════════════════════════════════════════════════════════════
 LEARNING OBJECTIVES
 ───────────────────
 1. Detect and handle missing (null) values using isnull, dropna, and fillna
 2. Find and remove duplicate rows using duplicated and drop_duplicates
 3. Convert column data types using astype, pd.to_datetime, and pd.to_numeric

 TIME:  ~30-35 minutes

 YOUTUBE
 ───────
 Search: "pandas data cleaning missing values tutorial"
 Search: "pandas dropna fillna duplicates astype"

══════════════════════════════════════════════════════════════

In [ ]:
import pandas as pd
import numpy as np
import os

# Load the Titanic dataset — it has real nulls to practice on
this_file = os.path.dirname(__file__)
titanic_path = os.path.join(this_file, "..", "datasets", "titanic_train.xlsx")

try:
    df = pd.read_excel(titanic_path)
    print("Titanic loaded:", df.shape)
except FileNotFoundError:
    print("titanic_train.xlsx not found. Using inline demo data.")
    # Inline fallback so the lesson works without the file
    df = pd.DataFrame({
        "PassengerId": range(1, 11),
        "Survived": [0, 1, 1, 1, 0, 0, 0, 1, 1, 0],
        "Pclass":   [3, 1, 3, 1, 3, 3, 1, 3, 2, 2],
        "Name":     ["Braund, Mr. Owen", "Cumings, Mrs. John", "Heikkinen, Miss. Laina",
                     "Futrelle, Mrs. Jacques", "Allen, Mr. William",
                     "Moran, Mr. James", "McCarthy, Mr. Timothy",
                     "Palsson, Master. Gosta", "Johnson, Mrs. Oscar", "Nasser, Mrs. Nicholas"],
        "Sex":      ["male","female","female","female","male","male","male","male","female","female"],
        "Age":      [22.0, 38.0, 26.0, 35.0, np.nan, np.nan, 54.0, 2.0, 27.0, 14.0],
        "Fare":     [7.25, 71.28, 7.93, 53.10, 8.05, 8.46, 51.86, 21.07, 11.13, 30.07],
        "Cabin":    [np.nan, "C85", np.nan, "C123", np.nan, np.nan, "E46", np.nan, np.nan, np.nan],
        "Embarked": ["S", "C", "S", "S", "S", "Q", "S", "S", "S", np.nan],
    })

══════════════════════════════════════════════════════════════
 CONCEPT 1 — FINDING AND HANDLING NULL VALUES (isnull, dropna, fillna)

══════════════════════════════════════════════════════════════
NULL values (NaN in pandas) appear whenever data is missing.
They break calculations: sum, mean, etc. silently skip them.
You must decide: drop the row, fill with a value, or leave it as-is.

DETECTION:
  df.isnull()              -- True/False mask for every cell
  df.isnull().sum()        -- count of nulls per column
  df.isnull().sum() / len(df) * 100  -- null percentage per column
  df.notnull()             -- opposite of isnull

HANDLING:
  df.dropna()              -- drop rows with ANY null
  df.dropna(subset=["Age"]) -- drop rows where Age is null only
  df.dropna(how="all")     -- drop rows where ALL values are null
  df.fillna(0)             -- replace all nulls with 0
  df.fillna({"Age": df["Age"].mean()}) -- fill each column differently
  df["col"].fillna(method="ffill") -- forward-fill (carry last known value)

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
print("\n=== NULL VALUE ANALYSIS ===")
print("Null count per column:")
print(df.isnull().sum())

print("\nNull percentage per column:")
null_pct = (df.isnull().sum() / len(df) * 100).round(1)
print(null_pct)

# Which columns have more than 10% nulls?
high_null_cols = null_pct[null_pct > 10].index.tolist()
print(f"\nColumns with >10% nulls: {high_null_cols}")

# Strategy 1: Fill Age with the median (more robust than mean for skewed data)
median_age = df["Age"].median()
df_filled = df.copy()
df_filled["Age"] = df_filled["Age"].fillna(median_age)
print(f"\nMedian age used for fill: {median_age}")
print(f"Age nulls after fill: {df_filled['Age'].isnull().sum()}")   # 0

# Strategy 2: Fill Embarked (categorical) with the most common value (mode)
most_common_port = df["Embarked"].mode()[0]
df_filled["Embarked"] = df_filled["Embarked"].fillna(most_common_port)
print(f"Embarked filled with: {most_common_port}")

# Strategy 3: Drop rows where the key target column is null
df_dropped = df.dropna(subset=["Survived"])
print(f"\nRows after dropna on Survived: {len(df_dropped)} (was {len(df)})")

# Strategy 4: Drop an entire column that is mostly null (Cabin is ~77% null)
if "Cabin" in df_filled.columns and null_pct.get("Cabin", 0) > 50:
    df_no_cabin = df_filled.drop(columns=["Cabin"])
    print(f"Dropped 'Cabin' column. Remaining columns: {len(df_no_cabin.columns)}")

══════════════════════════════════════════════════════════════
 EXERCISE 1

══════════════════════════════════════════════════════════════
Use the original df (with nulls).
Create a new DataFrame df_clean by:
  1. Filling null Ages with the MEAN age (round the mean to 1 decimal)
  2. Dropping any rows where Embarked is null
  3. Print how many rows were removed (before vs after)
  4. Print the null count per column to confirm no nulls remain
     (except Cabin which you are allowed to leave as-is)

Expected output:
  Original rows: 891
  After cleaning: 889
  Null counts after cleaning:
  ... (only Cabin should remain > 0 if present)

══════════════════════════════════════════════════════════════
 CONCEPT 2 — FINDING AND REMOVING DUPLICATES (duplicated, drop_duplicates)

══════════════════════════════════════════════════════════════
Duplicate rows cause inflated counts and incorrect aggregations.
pandas makes it easy to detect and remove them.

  df.duplicated()                -- True for every row that is a duplicate of a previous row
  df.duplicated(subset=["col"]) -- duplicates based on one column only
  df.duplicated().sum()          -- count of duplicate rows
  df.drop_duplicates()           -- return DataFrame with duplicates removed
  df.drop_duplicates(subset=["col"], keep="last") -- keep last occurrence

EXAMPLE ──────────────────────────────────────────────────────
Create a DataFrame with intentional duplicates to demonstrate

In [ ]:
orders_data = {
    "order_id":    ["ORD-001", "ORD-002", "ORD-003", "ORD-001", "ORD-004", "ORD-002"],
    "customer":    ["Acme",    "Beta",    "Gamma",   "Acme",    "Delta",   "Beta"],
    "product":     ["Widget",  "Gadget",  "Widget",  "Widget",  "Gadget",  "Gadget"],
    "amount":      [1200,      850,       700,       1200,      950,       850],
}

df_orders = pd.DataFrame(orders_data)
print("\n=== DUPLICATES ===")
print("Orders with duplicates:")
print(df_orders)

# Check for duplicates
print(f"\nDuplicate rows: {df_orders.duplicated().sum()}")   # 2

# See which rows are duplicates
print("\nDuplicate mask:")
print(df_orders[df_orders.duplicated(keep=False)])   # show ALL copies

# Remove complete duplicates
df_no_dups = df_orders.drop_duplicates()
print(f"\nAfter drop_duplicates: {len(df_no_dups)} rows (was {len(df_orders)})")
print(df_no_dups)

# Check duplicates on a single column (order_id should be unique)
dup_ids = df_orders[df_orders.duplicated(subset=["order_id"])]["order_id"]
print(f"\nDuplicate order IDs: {dup_ids.tolist()}")

# Keep the last occurrence when there are conflicting duplicates
df_keep_last = df_orders.drop_duplicates(subset=["order_id"], keep="last")
print("\nAfter dedup on order_id (keep last):")
print(df_keep_last)

══════════════════════════════════════════════════════════════
 EXERCISE 2

══════════════════════════════════════════════════════════════
The CRM export below has duplicate contact records.
Some contacts appear multiple times with slightly different entry dates.

Tasks:
  1. Count how many rows are full duplicates
  2. Count how many unique email addresses exist
  3. Remove duplicates keeping only the FIRST occurrence per email
  4. Print the cleaned DataFrame

Expected output:
  Full duplicate rows: 1
  Unique emails: 4
  Cleaned contacts (4 rows):
  [cleaned data]

In [ ]:
contacts = pd.DataFrame({
    "email":      ["alice@corp.com", "bob@corp.com", "carol@corp.com",
                   "bob@corp.com",   "dave@corp.com","alice@corp.com"],
    "name":       ["Alice Ng",       "Bob Chen",     "Carol Diaz",
                   "Bob Chen",       "Dave Park",    "Alice Ng"],
    "created":    ["2024-01-10",     "2024-01-12",   "2024-01-15",
                   "2024-01-12",     "2024-01-18",   "2024-01-10"],
})

══════════════════════════════════════════════════════════════
 CONCEPT 3 — CONVERTING DATA TYPES (astype, pd.to_datetime, pd.to_numeric)

══════════════════════════════════════════════════════════════
When pandas reads data from Excel or CSV, it sometimes assigns the wrong type.
Common issues:
  - Numbers stored as strings (e.g. "1,500" instead of 1500)
  - Dates stored as strings ("2024-01-15" instead of datetime)
  - Booleans stored as "True"/"False" strings

CONVERSION TOOLS:
  df["col"].astype(int)          -- convert to integer (fails on nulls/non-numeric)
  df["col"].astype(float)        -- convert to float
  df["col"].astype(str)          -- convert to string
  pd.to_numeric(df["col"])       -- smart conversion, errors="coerce" turns bad values to NaN
  pd.to_datetime(df["col"])      -- parse strings into datetime objects
  df["col"].astype("category")   -- memory-efficient for low-cardinality string columns

EXAMPLE ──────────────────────────────────────────────────────
Create a messy import to demonstrate

In [ ]:
raw_data = pd.DataFrame({
    "sale_date":  ["2024-01-15", "2024-02-22", "2024-03-08", "2024-04-01"],
    "amount":     ["1500",       "2,300",       "850",         "4100"],
    "qty":        ["10",         "15",          "7",           "22"],
    "is_online":  ["True",       "False",       "True",        "True"],
    "region":     ["West",       "East",        "West",        "Central"],
})

print("\n=== RAW DATA TYPES ===")
print(raw_data.dtypes)
# All columns are "object" (string) because they were read from a dirty source

# Convert sale_date to datetime
raw_data["sale_date"] = pd.to_datetime(raw_data["sale_date"])

# Convert amount: strip the comma first, then to numeric
raw_data["amount"] = pd.to_numeric(raw_data["amount"].str.replace(",", ""))

# Convert qty to integer
raw_data["qty"] = raw_data["qty"].astype(int)

# Convert is_online to boolean
raw_data["is_online"] = raw_data["is_online"].map({"True": True, "False": False})

# Convert region to category (saves memory when there are many repeated values)
raw_data["region"] = raw_data["region"].astype("category")

print("\n=== CLEANED DATA TYPES ===")
print(raw_data.dtypes)
print()
print(raw_data)

# Demonstrate pd.to_numeric with errors="coerce"
messy_amounts = pd.Series(["1500", "2300", "N/A", "850", "unknown", "4100"])
clean_amounts = pd.to_numeric(messy_amounts, errors="coerce")  # bad values become NaN
print("\nCleaned amounts (coerce):", clean_amounts.tolist())
# [1500.0, 2300.0, NaN, 850.0, NaN, 4100.0]

# Extract date parts after converting to datetime
raw_data["month"] = raw_data["sale_date"].dt.month
raw_data["year"]  = raw_data["sale_date"].dt.year
raw_data["day_name"] = raw_data["sale_date"].dt.day_name()
print("\nWith date parts extracted:")
print(raw_data[["sale_date", "month", "year", "day_name"]])

══════════════════════════════════════════════════════════════
 EXERCISE 3

══════════════════════════════════════════════════════════════
The DataFrame below arrived from an HR system export with bad data types.
Fix all of them so the final dtypes are correct.

Required final types:
  hire_date   -> datetime64
  salary      -> float64 (strip "$" and "," first)
  years_exp   -> int64 (strip " yrs")
  is_manager  -> bool (map "Yes"/"No" to True/False)
  department  -> category

After fixing, print:
  1. The dtypes
  2. The average salary by is_manager group

Expected output:
  hire_date      datetime64[ns]
  salary         float64
  years_exp      int64
  is_manager     bool
  department     category

  Average salary:
  is_manager
  False    76500.0
  True     98000.0

In [ ]:
hr_export = pd.DataFrame({
    "hire_date":  ["2020-03-15", "2018-07-01", "2022-11-20", "2019-05-08"],
    "salary":     ["$95,000",    "$72,000",    "$81,000",    "$88,000"],
    "years_exp":  ["8 yrs",      "12 yrs",     "3 yrs",      "7 yrs"],
    "is_manager": ["Yes",        "No",         "No",         "No"],
    "department": ["Engineering","Sales",      "Finance",    "Engineering"],
})

══════════════════════════════════════════════════════════════
 CONCEPT 4 — DATA QUALITY ASSERTIONS: FAIL FAST
══════════════════════════════════════════════════════════════
 ROYAL ROAD STANDARD — W4
 Run quality assertions before every transformation step.
 A dataset that fails a check should stop the pipeline immediately —
 not silently corrupt downstream output.

 assert statement:
   assert condition, "message if False"
   Raises AssertionError if condition is False.

 Pattern: write small, single-purpose assertion functions.
 Call them all before transformation begins. This is your data contract.

 EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
import pandas as pd


def assert_no_nulls(df, columns, label=""):
    for col in columns:
        n = df[col].isnull().sum()
        assert n == 0, f"[{label}] '{col}' has {n} null values"


def assert_value_range(df, column, min_val, max_val, label=""):
    out = df[(df[column] < min_val) | (df[column] > max_val)]
    assert len(out) == 0, (
        f"[{label}] '{column}' has {len(out)} values outside [{min_val}, {max_val}]"
    )


def assert_unique(df, column, label=""):
    dups = df[column].duplicated().sum()
    assert dups == 0, f"[{label}] '{column}' has {dups} duplicate values"


orders = pd.DataFrame({
    "order_id": ["ORD-001", "ORD-002", "ORD-003"],
    "amount":   [1200.0, 850.0, 700.0],
    "quantity": [3, 2, 5],
})

assert_no_nulls(orders, ["order_id", "amount", "quantity"], label="orders_load")
assert_value_range(orders, "amount", 0.01, 100_000, label="orders_load")
assert_unique(orders, "order_id", label="orders_load")
print("All quality checks passed.")

# Introduce a bad row and catch the failure
bad_orders = orders.copy()
bad_orders.loc[3] = ["ORD-001", None, 2]

try:
    assert_no_nulls(bad_orders, ["amount"], label="orders_load")
except AssertionError as e:
    print(f"Quality check failed: {e}")

══════════════════════════════════════════════════════════════
 EXERCISE 4
══════════════════════════════════════════════════════════════
 The inventory DataFrame below has intentional data errors.
 Write quality assertions for these rules and call them:
   1. No nulls in: product_id, name, price, quantity
   2. price must be between 0.01 and 50000
   3. quantity must be >= 0
   4. product_id must be unique

 At least one assertion will fail. When it does:
   - Catch the AssertionError and print the message
   - Fix the bad row in the DataFrame
   - Re-run all assertions to confirm they all pass
   - Print "All quality checks passed."

 Expected output:
     Quality check failed: [inventory] 'price' has 1 values outside [0.01, 50000]
     Quality check failed: [inventory] 'product_id' has 1 duplicate values
     All quality checks passed.

 --- starting data ---

In [ ]:
inventory = pd.DataFrame({
    "product_id": ["P001", "P002", "P003", "P002"],
    "name":       ["Laptop", "Monitor", "Keyboard", "Monitor"],
    "price":      [4500.0, 350.0, -1.0, 350.0],
    "quantity":   [10, 25, 50, 25],
})

# ══════════════════════════════════════════════════════════════
#  CONCEPT 5 -- PARQUET: THE STANDARD FORMAT FOR ANALYTICS DATA
# ══════════════════════════════════════════════════════════════
#
#  CSV stores data row-by-row as plain text. Every read scans every
#  column even if you only need two. For large datasets this is slow.
#
#  Parquet is a columnar binary format:
#    - Only the columns you SELECT are read from disk
#    - Built-in compression (typically 5-10x smaller than CSV)
#    - Preserves data types (no need to re-parse strings to floats)
#    - The standard output format for ETL pipelines and cloud storage
#
#  Production stacks: Spark, AWS Athena, BigQuery, DuckDB all read
#  Parquet natively. Your pipeline output should always be Parquet.
#
#  pip install pyarrow
#
#  EXAMPLE ----------------------------------------------------------

In [ ]:
import pandas as pd
import os
import tempfile

# Build a sample DataFrame
sales_data = {
    'month':   ['Jan', 'Feb', 'Mar', 'Apr', 'May'],
    'region':  ['North', 'North', 'South', 'South', 'East'],
    'revenue': [120000, 134000, 98000, 112000, 143000],
    'units':   [240, 268, 196, 224, 286],
}
df = pd.DataFrame(sales_data)

print('Original DataFrame:')
print(df)
print(f'\ndtypes:\n{df.dtypes}')

# Save as Parquet
parquet_path = os.path.join(tempfile.gettempdir(), 'sales.parquet')
df.to_parquet(parquet_path, index=False)

# Reload
df_loaded = pd.read_parquet(parquet_path)
print(f'\nReloaded from Parquet -- rows: {len(df_loaded)}, cols: {list(df_loaded.columns)}')
print(f'Types preserved: revenue={df_loaded["revenue"].dtype}, units={df_loaded["units"].dtype}')

# File size comparison
csv_path = parquet_path.replace('.parquet', '.csv')
df.to_csv(csv_path, index=False)
parquet_size = os.path.getsize(parquet_path)
csv_size     = os.path.getsize(csv_path)
print(f'\nFile sizes -- Parquet: {parquet_size} bytes, CSV: {csv_size} bytes')
print('(Parquet compression advantage grows with larger datasets)')

# ══════════════════════════════════════════════════════════════
#  EXERCISE 5
# ══════════════════════════════════════════════════════════════
#
#  A data pipeline receives a raw orders CSV with messy data.
#  Clean it and save the result as Parquet.
#
#  Steps:
#    1. Load the raw_orders dict into a DataFrame
#    2. Drop rows where 'amount' is None
#    3. Strip whitespace from 'customer' names
#    4. Convert 'date' to datetime using pd.to_datetime()
#    5. Save to tempfile.gettempdir() / 'clean_orders.parquet'
#    6. Reload from the Parquet file and verify: row count = 4
#
#  Expected output:
#    Cleaned rows  : 4
#    Reloaded rows : 4
#    Columns       : ['order_id', 'customer', 'amount', 'date']
#    Types         : amount=float64, date=datetime64[ns]
#
# --- starting data ---

In [ ]:
import pandas as pd
import os
import tempfile

raw_orders = {
    'order_id': [101, 102, 103, 104, 105],
    'customer': ['  Alice', 'Bob  ', ' Carol', 'Dave', '  Eve'],
    'amount':   [250.0, None, 180.5, 320.0, 95.0],
    'date':     ['2024-01-05', '2024-01-08', '2024-01-10', '2024-01-12', '2024-01-15'],
}

══════════════════════════════════════════════════════════════
 CONCEPT 6 — TESTING YOUR DATA QUALITY FUNCTIONS
══════════════════════════════════════════════════════════════

 Shift-Left Testing applied to data pipelines: verify that your quality
 assertion functions behave correctly before you depend on them.

 You need to test two things:
   1. They PASS silently on clean data (no exception raised)
   2. They RAISE AssertionError on dirty data (and the message is useful)

 Pattern for testing that an error is raised:
   error_raised = False
   try:
       assert_function(bad_data, ...)
   except AssertionError:
       error_raised = True
   assert error_raised, "function must catch this specific violation"

 A quality function that does not raise on bad data is worse than useless —
 it gives you false confidence. Always test both the pass and fail paths.

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
import pandas as pd

# assert_no_nulls, assert_value_range, assert_unique are defined in CONCEPT 4 above.

clean_df = pd.DataFrame({
    "order_id": ["O1", "O2", "O3"],
    "amount":   [100.0, 250.0, 80.0],
    "qty":      [2, 5, 1],
})

# 1. Assert clean data passes silently
assert_no_nulls(clean_df, ["order_id", "amount", "qty"],  label="clean_check")
assert_value_range(clean_df, "amount", 0.01, 10_000,      label="clean_check")
assert_unique(clean_df, "order_id",                        label="clean_check")

# 2. Assert dirty data is caught — null in amount
df_with_null = clean_df.copy()
df_with_null.loc[3] = ["O4", None, 1]
null_caught = False
try:
    assert_no_nulls(df_with_null, ["amount"], label="null_test")
except AssertionError:
    null_caught = True
assert null_caught, "assert_no_nulls must raise on null values"

# 3. Assert out-of-range amount is caught
df_negative = clean_df.copy()
df_negative.loc[3] = ["O4", -50.0, 1]
range_caught = False
try:
    assert_value_range(df_negative, "amount", 0.01, 10_000, label="range_test")
except AssertionError:
    range_caught = True
assert range_caught, "assert_value_range must raise on out-of-range values"

# 4. Assert duplicate order_id is caught
df_dup = pd.concat([clean_df, clean_df.iloc[[0]]], ignore_index=True)
dup_caught = False
try:
    assert_unique(df_dup, "order_id", label="dup_test")
except AssertionError:
    dup_caught = True
assert dup_caught, "assert_unique must raise on duplicate values"

print("Data quality function contract: all assertions passed.")

══════════════════════════════════════════════════════════════
 EXERCISE 6
══════════════════════════════════════════════════════════════

 The three quality functions and the shipments DataFrame are in the
 starting data below.

 Write assert statements to verify:
   1. assert_no_nulls passes on clean columns: ["shipment_id", "weight_kg", "status"]
   2. assert_value_range passes: weight_kg between 0.1 and 5000
   3. assert_unique passes: shipment_id is unique
   4. assert_no_nulls raises when "carrier" column has a null
   5. assert_value_range raises when weight_kg has a value of -10.0

 Use the error_raised = False / try / except pattern for checks 4 and 5.
 The last line must print: "Shipments quality contract: all assertions passed."

 Expected output:
     Shipments quality contract: all assertions passed.

 --- starting data ---

In [ ]:
import pandas as pd
import numpy as np


def assert_no_nulls(df, columns, label=""):
    for col in columns:
        n = df[col].isnull().sum()
        assert n == 0, f"[{label}] '{col}' has {n} null values"


def assert_value_range(df, column, min_val, max_val, label=""):
    out = df[(df[column] < min_val) | (df[column] > max_val)]
    assert len(out) == 0, (
        f"[{label}] '{column}' has {len(out)} values outside [{min_val}, {max_val}]"
    )


def assert_unique(df, column, label=""):
    dups = df[column].duplicated().sum()
    assert dups == 0, f"[{label}] '{column}' has {dups} duplicate values"


shipments = pd.DataFrame({
    "shipment_id": ["SH-001", "SH-002", "SH-003"],
    "weight_kg":   [12.5, 340.0, 8.0],
    "status":      ["delivered", "in_transit", "delivered"],
    "carrier":     ["FedEx", "UPS", np.nan],
})

shipments_bad_weight = pd.DataFrame({
    "shipment_id": ["SH-004"],
    "weight_kg":   [-10.0],
    "status":      ["pending"],
    "carrier":     ["DHL"],
})




